# YOLOv8 Model Downloader, Benchmark & Performance Selector

This Jupyter Notebook helps you **download, benchmark, and select** the best-performing model for your traffic monitoring setup.

### In this notebook:
1. **Model Download Hub**: Download YOLOv8 variants (`yolov8n`, `yolov8s`, `yolov8m`, `yolov8l`).
2. **Model Specs & Architectural Comparison**: Parameters, FLOPs, and theoretical latency.
3. **Inference Latency & FPS Benchmark**: Test each model on your local hardware (CPU / GPU).
4. **Export for Peak Performance**: Export the chosen model to **ONNX / TensorRT / OpenVINO**.
5. **One-Click Deploy**: Update your `.env` and `config.yaml` with the winning model.

In [1]:
# Step 0: Environment & Hardware Check
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from ultralytics import YOLO

print("=== System Hardware Inspection ===")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA (GPU Acceleration) Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running in CPU mode. Lightweight models (yolov8n, yolov8s) are strongly recommended.")


=== System Hardware Inspection ===
PyTorch: 2.14.0+cpu
CUDA (GPU Acceleration) Available: False
Running in CPU mode. Lightweight models (yolov8n, yolov8s) are strongly recommended.


---
## Step 1: Model Catalog & Specifications

| Model | Size (MB) | mAP50-95 (COCO) | Parameters (M) | Ideal Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **YOLOv8n** (Nano) | ~6.5 MB | 37.3 | 3.2M | Ultra-high FPS, edge devices, CPUs, real-time multi-cam |
| **YOLOv8s** (Small) | ~22.5 MB | 44.9 | 11.2M | Best balance of speed and detection accuracy |
| **YOLOv8m** (Medium) | ~52.0 MB | 50.2 | 25.9M | High accuracy for distant or small vehicles, needs dedicated GPU |
| **YOLOv8l** (Large) | ~87.0 MB | 52.9 | 43.7M | Maximum accuracy for offline video analytics |

In [2]:
# Step 2: Download Models to Local Storage
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd().resolve()
WEIGHTS_DIR = PROJECT_ROOT / "models" / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Select which models to download
models_to_download = ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]
loaded_models = {}

for model_name in models_to_download:
    target_path = WEIGHTS_DIR / model_name
    print(f"\nDownloading / Loading {model_name} into {target_path.relative_to(PROJECT_ROOT)}...")
    model = YOLO(model_name)
    # Save locally to models/weights if downloaded to root
    if Path(model_name).exists() and not target_path.exists():
        import shutil
        shutil.copy(model_name, str(target_path))
    loaded_models[model_name] = model
    size_mb = os.path.getsize(target_path) / (1024 * 1024) if target_path.exists() else 0
    print(f"[Ready] {model_name} - Size: {size_mb:.2f} MB")



[Ready] yolov8n.pt - Size: 6.25 MB

[Ready] yolov8s.pt - Size: 21.54 MB

[Ready] yolov8m.pt - Size: 49.72 MB


---
## Step 3: Local Performance & Latency Benchmark

We benchmark each downloaded model on standard 1080p traffic frames to measure:
* **Average Latency (ms)**
* **Real-Time Throughput (FPS)**
* **Memory Consumption**

In [3]:
# Benchmark Configuration
BENCHMARK_FRAMES = 20
DUMMY_FRAME = np.random.randint(0, 255, (1080, 1920, 3), dtype=np.uint8)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

results = []
print(f"Running benchmark on device: {DEVICE} ({BENCHMARK_FRAMES} frames per model)\n")

for name, model in loaded_models.items():
    # Warm-up pass
    for _ in range(3):
        _ = model.predict(DUMMY_FRAME, imgsz=640, device=DEVICE, verbose=False)
    
    # Timed inference
    latencies = []
    for _ in range(BENCHMARK_FRAMES):
        t0 = time.perf_counter()
        _ = model.predict(DUMMY_FRAME, imgsz=640, device=DEVICE, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)
    
    avg_ms = float(np.mean(latencies))
    min_ms = float(np.min(latencies))
    fps = 1000.0 / avg_ms
    target_file = WEIGHTS_DIR / name
    size_mb = os.path.getsize(target_file) / (1024 * 1024) if target_file.exists() else 0
    
    results.append({
        "Model": name,
        "Size (MB)": round(size_mb, 2),
        "Avg Latency (ms)": round(avg_ms, 2),
        "Min Latency (ms)": round(min_ms, 2),
        "FPS": round(fps, 1),
        "Real-Time (>30 FPS)": "YES" if fps >= 30 else "NO"
    })

df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))


Running benchmark on device: cpu (20 frames per model)

| Model      |   Size (MB) |   Avg Latency (ms) |   Min Latency (ms) |   FPS | Real-Time (>30 FPS)   |
|:-----------|------------:|-------------------:|-------------------:|------:|:----------------------|
| yolov8n.pt |        6.25 |              72.81 |              57.83 |  13.7 | NO                    |
| yolov8s.pt |       21.54 |             206.01 |             165.58 |   4.9 | NO                    |
| yolov8m.pt |       49.72 |             493.35 |             427.21 |   2   | NO                    |


---
## Step 4: Export Best Model for Maximum Performance

Choose your optimal model based on the benchmark table and export it for 2x to 3x higher throughput:
* **ONNX (with FP16)**: Universal accelerator for CPU & GPU.
* **TensorRT (`.engine`)**: For NVIDIA GPUs.
* **OpenVINO**: For Intel CPUs.

In [5]:
pip install onnx

Defaulting to user installation because normal site-packages is not writeable
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/7.9 MB 1.7 MB/s eta 0:00:05
   ------ --------------------------------- 1.3/7.9 MB 2.1 MB/s eta 0:00:04
   --------- ------------------------------ 1.8/7.9 MB 2.3 MB/s eta 0:00:03
   ------------- -------------------------- 2.6/7.9 MB 2.6 MB/s eta 0:00:03
   ------------------ --------------------- 3.7/7.9 MB 3.0 MB/s eta 0:00:02
   ----------------------- ---------------- 4.7/7.9 MB 3.2 MB/s eta 0:00:01
   -------------------------- ------------- 5.2/7.9 MB 3.3 MB/s eta 0:00:01
   ------------------------------ --------- 6.0/7.9 MB 3.3 MB/s eta 0:00:01
   ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [6]:
# Select the best model for export (e.g. yolov8s.pt or yolov8n.pt)
SELECTED_MODEL = "yolov8n.pt"  # Change to "yolov8s.pt" if you have a GPU
best_model = YOLO(str(WEIGHTS_DIR / SELECTED_MODEL))

print(f"Optimizing and exporting {SELECTED_MODEL}...")

# Export to ONNX with half-precision (FP16) if on GPU, or FP32 on CPU
use_fp16 = torch.cuda.is_available()
exported_file = best_model.export(format="onnx", half=use_fp16, simplify=True)
print(f"\n[Success] Optimized model exported to: {exported_file}")


Optimizing and exporting yolov8n.pt...
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.171  Python-3.13.9 torch-2.14.0+cpu CPU (11th Gen Intel Core i3-1115G4 @ 3.00GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'D:\INTELLIGENT_TRAFFIC_MONITORING\models\weights\yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'C:\\ProgramData\\anaconda3\\python.exe', 'onnxruntime', 'onnxslim>=0.1.82', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 2.
WARNING Retry 2/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'C:\\ProgramData\\anaconda3\\python.exe', 'onnxruntime', '

---
## Step 5: Update Project Configuration with Best Model

Save your chosen model path directly into `.env` so your traffic monitoring application uses it immediately.

In [7]:
# Update .env file with the selected model
env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    with open(env_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    new_lines = []
    updated = False
    for line in lines:
        if line.startswith("MODEL_NAME="):
            new_lines.append(f"MODEL_NAME={SELECTED_MODEL}\n")
            updated = True
        else:
            new_lines.append(line)
    
    if not updated:
        new_lines.append(f"\nMODEL_NAME={SELECTED_MODEL}\n")
        
    with open(env_path, "w", encoding="utf-8") as f:
        f.writelines(new_lines)
        
    print(f"[Config Updated] .env is now configured to use: {SELECTED_MODEL}")
else:
    print(f".env not found at {env_path}")


[Config Updated] .env is now configured to use: yolov8n.pt
